# Uniform-`n` partial mantissa-protection results

This is the single notebook for all graphs generated from the validated 10,000-image uniform active-mantissa protection experiment using LeNeT/MNIST. It reads the preserved summary table and does not regenerate or modify the experimental results.

The Python kernel requires `pandas`, `numpy`, and `matplotlib`.

## Experiment and interpretation scope

The plotted classification is the simulator's **effective type by instruction**. The sweep has three saturation regimes:

- `n = 0..8`: FP64, FP32, and FP16 protection change;
- `n = 9..21`: FP64 and FP32 protection change;
- `n = 22..50`: only FP64 protection changes.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import PercentFormatter, StrMethodFormatter

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing AGENTS.md")


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


REPOSITORY_ROOT = find_repository_root(Path.cwd())
CAMPAIGN_DIRECTORY = (
    REPOSITORY_ROOT
    / "verification/out/lenet-uniform-n-full-matrix-e85378fa-v1"
)
SUMMARY_CSV = CAMPAIGN_DIRECTORY / "summary/summary.csv"
REGIONS_CSV = CAMPAIGN_DIRECTORY / "summary/regions.csv"
SUMMARY_MANIFEST = CAMPAIGN_DIRECTORY / "summary/summary-manifest.json"
MATRIX_GATE = CAMPAIGN_DIRECTORY / "matrix-gate.json"

# Set to False when notebook execution should remain read-only.
SAVE_FIGURES = True
FIGURE_DIRECTORY = REPOSITORY_ROOT / "verification/out/uniform-n-results-figures"

for required_path in (SUMMARY_CSV, REGIONS_CSV, SUMMARY_MANIFEST, MATRIX_GATE):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required campaign artifact not found: {required_path}")

print(f"Repository: {REPOSITORY_ROOT}")
print(f"Campaign:   {CAMPAIGN_DIRECTORY}")

## Validate and load the campaign summary

The checks below prevent plotting an incomplete matrix, a failed campaign gate, or a summary CSV that no longer matches its recorded hash.

In [ ]:
with MATRIX_GATE.open(encoding="utf-8") as stream:
    gate = json.load(stream)
with SUMMARY_MANIFEST.open(encoding="utf-8") as stream:
    summary_manifest = json.load(stream)

if gate.get("status") != "passed":
    raise ValueError(f"Campaign matrix gate is not passed: {gate.get('status')!r}")
if gate.get("configuration_count") != 51:
    raise ValueError("Expected exactly 51 validated configurations")

expected_summary_hash = summary_manifest["artifacts"]["summary.csv"]
observed_summary_hash = sha256(SUMMARY_CSV)
if observed_summary_hash != expected_summary_hash:
    raise ValueError("summary.csv does not match summary-manifest.json")
expected_regions_hash = summary_manifest["artifacts"]["regions.csv"]
observed_regions_hash = sha256(REGIONS_CSV)
if observed_regions_hash != expected_regions_hash:
    raise ValueError("regions.csv does not match summary-manifest.json")

data = pd.read_csv(SUMMARY_CSV).sort_values("n").reset_index(drop=True)
TYPE_COLUMNS = {
    "E5M2": "effective_e5m2",
    "FP16": "effective_fp16",
    "FP32": "effective_fp32",
    "FP64": "effective_fp64",
}
required_columns = {
    "n",
    "processed",
    "accuracy",
    "effective_type_observations",
    *TYPE_COLUMNS.values(),
}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Summary CSV is missing columns: {sorted(missing_columns)}")
if data["n"].tolist() != list(range(51)):
    raise ValueError("Expected one ordered row for every n from 0 through 50")
if not data["processed"].eq(10_000).all():
    raise ValueError("At least one configuration did not process 10,000 images")

classified_sum = data[list(TYPE_COLUMNS.values())].sum(axis=1)
if not classified_sum.eq(data["effective_type_observations"]).all():
    raise ValueError("Per-type counts do not sum to effective_type_observations")

print("Campaign gate: passed")
print(f"Validated configurations: {len(data)}")
print(f"summary.csv SHA-256: {observed_summary_hash}")
print(f"regions.csv SHA-256: {observed_regions_hash}")

In [ ]:
display_columns = ["n", "accuracy", "effective_type_observations", *TYPE_COLUMNS.values()]
data.loc[data["n"].isin([0, 1, 8, 21, 50]), display_columns].rename(
    columns={"accuracy": "Accuracy", **{column: label for label, column in TYPE_COLUMNS.items()}}
)

## Classification accuracy versus `n`

Each bar represents the aggregate classification accuracy over the same 10,000 MNIST test images. The highlighted configurations are the no-protection endpoint (`n=0`), the FP16 saturation point (`n=8`), the FP32 saturation point (`n=21`), and the full-protection endpoint (`n=50`). These markers identify policy regimes; they do not isolate the causal effect of one floating-point format.

In [ ]:
HIGHLIGHTED_N = (0, 1, 3, 5, 8, 21, 50)
accuracy_percent = 100.0 * data["accuracy"]
is_highlighted = data["n"].isin(HIGHLIGHTED_N)
bar_colors = np.where(is_highlighted, "#F58518", "#4C78A8")
bar_edges = np.where(is_highlighted, "none", "none")
bar_linewidths = np.where(is_highlighted, 1.4, 0.0)

# Sized for the SBC template's 15 cm single-column text area.
fig, ax = plt.subplots(figsize=(7.0, 3.5), dpi=250)
bars = ax.bar(
    data["n"],
    accuracy_percent,
    width=0.88,
    color=bar_colors,
    edgecolor=bar_edges,
    linewidth=bar_linewidths,
)
for n in HIGHLIGHTED_N:
    index = data.index[data["n"].eq(n)].item()
    bar = bars[index]
    value = accuracy_percent.iloc[index]
    ax.annotate(
        # f"n={n}\n{value:.2f}%",
        f"{value:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        va="bottom",
        rotation=60,
        fontsize=7.5,
        fontweight="bold",
    )

ax.set(
    # title="LeNet classification accuracy vs protection of exceeding mantissa-bit"
    # title="",
    xlabel="mantissa-bit protection parameter (n)",
    ylabel="Accuracy (%)",
    xlim=(-0.75, 50.75),
    ylim=(0, 112),
)

ax.xaxis.label.set_size(11)
ax.yaxis.label.set_size(11)
ax.tick_params(axis="both", labelsize=9)

accuracy_x_ticks = (set(range(0, 51, 5)) - {20}) | set(HIGHLIGHTED_N)
ax.set_xticks(sorted(accuracy_x_ticks))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
fig.tight_layout()

if SAVE_FIGURES:
    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURE_DIRECTORY / "accuracy-vs-uniform-n.png",
        dpi=300,
        bbox_inches="tight",
    )
plt.show()

## Relative effective-type instruction observations

Each bar is normalized by the total number of classified effective-type observations for the same $n$. E5M2, FP16, FP32, and FP64 are simulator-assigned **effective types**, not architectural instruction widths. The counts represent dynamic instruction observations, not static instructions or register writes. `UNCLASSIFIED` is omitted because its count is zero for every validated configuration.

In [ ]:
TYPE_COLORS = {
    "E5M2": "#4C78A8",
    "FP16": "#F58518",
    "FP32": "#54A24B",
    "FP64": "#E45756",
}

effective_type_shares = data[list(TYPE_COLUMNS.values())].div(
    data["effective_type_observations"], axis=0
)
if not np.isfinite(effective_type_shares.to_numpy()).all():
    raise ValueError("Effective-type shares contain non-finite values.")
if (effective_type_shares.to_numpy() < 0).any():
    raise ValueError("Effective-type shares contain negative values.")
if not np.allclose(
    effective_type_shares.sum(axis=1).to_numpy(), 1.0, rtol=0.0, atol=1e-12
):
    raise ValueError("Effective-type shares do not sum to 100% for every n.")

fig, ax = plt.subplots(figsize=(7.0, 3.5), dpi=250)
bottom = np.zeros(len(data))
for label, column in TYPE_COLUMNS.items():
    values = effective_type_shares[column].to_numpy()
    ax.bar(
        data["n"], values, bottom=bottom, width=0.88,
        color=TYPE_COLORS[label], label=label, linewidth=0,
    )
    bottom += values

for boundary in (8.5, 21.5):
    ax.axvline(boundary, color="black", linestyle="--", linewidth=0.9, alpha=0.7)

effective_type_x_ticks = sorted((set(range(0, 51, 5)) - {20}) | {0, 8, 21, 50})
ax.set_xlabel("Mantissa-bit protection parameter (n)", fontsize=11)
ax.set_ylabel("Type distribution (%)", fontsize=11)
ax.set_xticks(effective_type_x_ticks)
ax.set_xlim(-0.75, 50.75)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax.tick_params(axis="both", labelsize=9)
ax.legend(
    title="Types", ncols=4, loc="lower center",
    bbox_to_anchor=(0.5, 1.01), frameon=False, fontsize=8, title_fontsize=8,
)
fig.tight_layout()
if SAVE_FIGURES:
    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURE_DIRECTORY / "effective-type-shares-vs-uniform-n.png",
        dpi=300, bbox_inches="tight",
    )
plt.show()

## Absolute-count view grouped by execution region

This alternative preserves the same absolute counts and selected $n=1,8,21$ configurations while reducing the presentation to two axes. Within each execution region, three adjacent horizontal stacked bars show the effective-type composition for the selected configurations. The bar-end labels identify $n$ directly. High- and lower-volume regions retain separate, explicitly labeled linear scales; bar totals are omitted to reduce visual load.

In [ ]:
GROUPED_SELECTED_N = (1, 8, 21)
GROUPED_REGION_GROUPS = (
    (["conv1-block", "conv2-block", "fc1"], 1_100, "High-volume layers (> 100M)"),
    (["unscoped", "input-normalization", "fc2", "softmax"], 30, "Lower-volume layers (< 30M)"),
)
GROUPED_REGION_LABELS = {
    "unscoped": "Unscoped",
    "input-normalization": "Input normalization",
    "conv1-block": "Conv1",
    "conv2-block": "Conv2",
    "fc1": "FC1",
    "fc2": "FC2",
    "softmax": "Softmax",
}
grouped_regions = pd.read_csv(REGIONS_CSV).sort_values(["n", "region_id"]).reset_index(drop=True)
grouped_region_names = {name for group, _, _ in GROUPED_REGION_GROUPS for name in group}
grouped_counts = grouped_regions.loc[
    grouped_regions["n"].isin(GROUPED_SELECTED_N)
    & grouped_regions["region_name"].isin(grouped_region_names)
].copy()
expected_grouped_rows = len(GROUPED_SELECTED_N) * len(grouped_region_names)
if len(grouped_counts) != expected_grouped_rows:
    raise ValueError("Expected exactly one row per selected (n, region) pair.")
if grouped_counts[list(TYPE_COLUMNS.values())].lt(0).any().any():
    raise ValueError("Grouped region counts contain negative values.")

for n in GROUPED_SELECTED_N:
    plotted_aggregate = (
        grouped_counts.loc[grouped_counts["n"].eq(n), list(TYPE_COLUMNS.values())].sum(axis=0)
    )
    expected_aggregate = data.set_index("n").loc[n, list(TYPE_COLUMNS.values())]
    if not plotted_aggregate.equals(expected_aggregate):
        raise ValueError(f"Grouped regions do not reconstruct aggregate counts for n={n}.")

fig, axes = plt.subplots(
    2, 1, figsize=(7.0, 7.0), dpi=250, constrained_layout=True,
    gridspec_kw={"height_ratios": (3, 4)},
)
bar_step = 0.72
group_gap = 0.65
for ax, (region_group, x_limit, panel_title) in zip(axes, GROUPED_REGION_GROUPS):
    bar_positions = []
    group_centers = []
    next_position = 0.0
    for region_name in region_group:
        positions = next_position + bar_step * np.arange(len(GROUPED_SELECTED_N))
        bar_positions.extend(positions)
        group_centers.append(positions.mean())
        next_position = positions[-1] + bar_step + group_gap

    left = np.zeros(len(bar_positions))
    ordered_rows = []
    for region_name in region_group:
        for n in GROUPED_SELECTED_N:
            match = grouped_counts.loc[
                grouped_counts["region_name"].eq(region_name) & grouped_counts["n"].eq(n)
            ]
            if len(match) != 1:
                raise ValueError(f"Missing or duplicate grouped count for n={n}, region={region_name}.")
            ordered_rows.append(match.iloc[0])
    ordered_counts = pd.DataFrame(ordered_rows).reset_index(drop=True)

    for label, column in TYPE_COLUMNS.items():
        values_millions = ordered_counts[column].to_numpy(dtype=float) / 1_000_000
        ax.barh(
            bar_positions, values_millions, left=left, height=0.56,
            color=TYPE_COLORS[label], label=label, linewidth=0,
        )
        left += values_millions

    label_offset = 0.012 * x_limit
    for position, total, n in zip(
        bar_positions, left, np.tile(GROUPED_SELECTED_N, len(region_group))
    ):
        ax.text(total + label_offset, position, f"n={n}", va="center", ha="left", fontsize=7)
    ax.set_yticks(group_centers, [GROUPED_REGION_LABELS[name] for name in region_group])
    ax.invert_yaxis()
    ax.set_xlim(0, x_limit)
    ax.set_title(panel_title, fontsize=10, fontweight="bold", loc="left")
    ax.set_xlabel("Absolute types distribution (millions)", fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, title="Types", ncols=4,
    loc="outside upper center", frameon=False,
    fontsize=8.5, title_fontsize=8.5,
)

if SAVE_FIGURES:
    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURE_DIRECTORY / "absolute-effective-type-counts-grouped-by-region-selected-n.png",
        dpi=300, bbox_inches="tight",
    )
plt.show()

## Side-by-side alternative for the absolute-count view

This alternative places the high- and lower-volume panels side by side while preserving the same absolute counts, selected $n=1,8,21$ configurations, type colors, direct $n$ labels, and separate linear scales. It is exported independently and does not replace the top–bottom version.

In [ ]:
SIDE_BY_SIDE_SELECTED_N = (1, 8, 21)
SIDE_BY_SIDE_REGION_GROUPS = (
    (["conv1-block", "conv2-block", "fc1"], 1_100, "High-volume layers (> 100M)"),
    (["unscoped", "input-normalization", "fc2", "softmax"], 30, "Lower-volume layers (< 30M)"),
)
SIDE_BY_SIDE_REGION_LABELS = {
    "unscoped": "Others",
    "input-normalization": "Input normalization",
    "conv1-block": "Conv1",
    "conv2-block": "Conv2",
    "fc1": "FC1",
    "fc2": "FC2",
    "softmax": "Softmax",
}
side_by_side_regions = pd.read_csv(REGIONS_CSV).sort_values(["n", "region_id"]).reset_index(drop=True)
side_by_side_region_names = {
    name for group, _, _ in SIDE_BY_SIDE_REGION_GROUPS for name in group
}
side_by_side_counts = side_by_side_regions.loc[
    side_by_side_regions["n"].isin(SIDE_BY_SIDE_SELECTED_N)
    & side_by_side_regions["region_name"].isin(side_by_side_region_names)
].copy()
expected_side_by_side_rows = len(SIDE_BY_SIDE_SELECTED_N) * len(side_by_side_region_names)
if len(side_by_side_counts) != expected_side_by_side_rows:
    raise ValueError("Expected exactly one row per selected (n, region) pair.")
if side_by_side_counts[list(TYPE_COLUMNS.values())].lt(0).any().any():
    raise ValueError("Side-by-side region counts contain negative values.")
for n in SIDE_BY_SIDE_SELECTED_N:
    plotted_aggregate = side_by_side_counts.loc[
        side_by_side_counts["n"].eq(n), list(TYPE_COLUMNS.values())
    ].sum(axis=0)
    expected_aggregate = data.set_index("n").loc[n, list(TYPE_COLUMNS.values())]
    if not plotted_aggregate.equals(expected_aggregate):
        raise ValueError(f"Side-by-side regions do not reconstruct aggregate counts for n={n}.")

fig, axes = plt.subplots(
    1, 2, figsize=(8.8, 5.2), dpi=250, constrained_layout=True,
    gridspec_kw={"width_ratios": (1.05, 1)},
)
bar_step = 0.72
group_gap = 0.65
for ax, (region_group, x_limit, panel_title) in zip(axes, SIDE_BY_SIDE_REGION_GROUPS):
    bar_positions = []
    group_centers = []
    next_position = 0.0
    for region_name in region_group:
        positions = next_position + bar_step * np.arange(len(SIDE_BY_SIDE_SELECTED_N))
        bar_positions.extend(positions)
        group_centers.append(positions.mean())
        next_position = positions[-1] + bar_step + group_gap

    ordered_rows = []
    for region_name in region_group:
        for n in SIDE_BY_SIDE_SELECTED_N:
            match = side_by_side_counts.loc[
                side_by_side_counts["region_name"].eq(region_name)
                & side_by_side_counts["n"].eq(n)
            ]
            if len(match) != 1:
                raise ValueError(f"Missing or duplicate side-by-side count for n={n}, region={region_name}.")
            ordered_rows.append(match.iloc[0])
    ordered_counts = pd.DataFrame(ordered_rows).reset_index(drop=True)
    left = np.zeros(len(bar_positions))
    for label, column in TYPE_COLUMNS.items():
        values_millions = ordered_counts[column].to_numpy(dtype=float) / 1_000_000
        ax.barh(
            bar_positions, values_millions, left=left, height=0.56,
            color=TYPE_COLORS[label], label=label, linewidth=0,
        )
        left += values_millions

    label_offset = 0.012 * x_limit
    for position, total, n in zip(
        bar_positions, left, np.tile(SIDE_BY_SIDE_SELECTED_N, len(region_group))
    ):
        ax.text(total + label_offset, position, f"n={n}", va="center", ha="left", fontsize=7)
    ax.set_yticks(group_centers, [SIDE_BY_SIDE_REGION_LABELS[name] for name in region_group])
    ax.invert_yaxis()
    ax.set_xlim(0, x_limit)
    ax.set_title(panel_title, fontsize=10, fontweight="bold", loc="left")
    ax.set_xlabel("Absolute types distribution (millions)", fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, title="Types", ncols=4,
    loc="outside upper center", frameon=False,
    fontsize=8.5, title_fontsize=8.5,
)

if SAVE_FIGURES:
    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURE_DIRECTORY / "absolute-effective-type-counts-grouped-by-region-selected-n-side-by-side.png",
        dpi=300, bbox_inches="tight",
    )
plt.show()